# Linguistic Agent Training (Whisper + BERT)

This notebook has two phases:
1. **Transcription**: Uses Whisper-small to convert the 40% audio dataset into text transcripts.
2. **Classification**: Fine-tunes a BERT model on those transcripts to classify Fake vs Real.

In [ ]:
!pip install transformers datasets evaluate accelerate torch torchaudio pandas scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
import pandas as pd
import torch

DRIVE_DIR = Path('/content/drive/MyDrive/40_PER_22_Data')
TRANSCRIPTS_CSV = DRIVE_DIR / 'transcripts.csv'
MODEL_SAVE_DIR = DRIVE_DIR / 'linguistic_bert_model'

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Phase 1: Whisper Transcription

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm

# Only run this if we haven't already generated the transcripts
if not TRANSCRIPTS_CSV.exists():
    print("Loading Whisper...")
    transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-small", device=0 if device=="cuda" else -1)
    
    # Collect audio files
    audio_files = []
    for split in ['train', 'test']:
        for cls, label in [('bonafide', 0), ('spoof', 1)]:
            folder = DRIVE_DIR / split / cls
            if folder.exists():
                for f in folder.glob('*.flac'):
                    audio_files.append((str(f), label, split))
                    
    print(f"Found {len(audio_files)} audio files to transcribe.")
    
    results = []
    for path, label, split in tqdm(audio_files):
        try:
            text = transcriber(path)["text"].strip()
            results.append({"filename": Path(path).name, "text": text, "label": label, "split": split})
        except Exception as e:
            pass
            
    df_trans = pd.DataFrame(results)
    df_trans.to_csv(TRANSCRIPTS_CSV, index=False)
    print(f"Saved {len(df_trans)} transcripts to {TRANSCRIPTS_CSV}")
else:
    print(f"Transcripts already exist at {TRANSCRIPTS_CSV}, skipping transcription phase.")

## Phase 2: BERT Fine-Tuning

In [ ]:
df = pd.read_csv(TRANSCRIPTS_CSV)
df = df.dropna(subset=['text'])
df = df[df['text'].str.strip().astype(bool)] # Remove empty strings

train_df = df[df['split'] == 'train']
test_df = df[df['split'] == 'test']

print(f"Train size: {len(train_df)}\nTest size: {len(test_df)}")

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset = train_dataset.remove_columns(['filename', 'text', 'split', '__index_level_0__'])
train_dataset = train_dataset.rename_column("label", "labels")
train_dataset.set_format("torch")

test_dataset = test_dataset.remove_columns(['filename', 'text', 'split', '__index_level_0__'])
test_dataset = test_dataset.rename_column("label", "labels")
test_dataset.set_format("torch")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    auc = roc_auc_score(labels, probs)
    
    fpr, tpr, _ = roc_curve(labels, probs)
    fnr = 1 - tpr
    try:
        eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except:
        eer = np.mean(np.abs(fnr - fpr))
        
    return {"accuracy": acc, "f1": f1, "auc": auc, "eer": eer}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True if device == "cuda" else False, # Mixed precision for faster training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting BERT fine-tuning...")
trainer.train()

In [ ]:
print(f"Saving fine-tuned model to {MODEL_SAVE_DIR}...")
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

eval_results = trainer.evaluate()
print("\nTest Set Results:")
print(f"Accuracy: {eval_results['eval_accuracy']*100:.2f}%")
print(f"EER:      {eval_results['eval_eer']*100:.2f}%")
print(f"AUC:      {eval_results['eval_auc']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")